In [1]:
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype
from collections import defaultdict

In [3]:
def preprocess(df):
    mapping = {'L사': 0, 'K사': 1, 'S사': 2}
    if '가입통신회사코드' in df.columns:
        df['가입통신회사코드'] = df['가입통신회사코드'].map(mapping).astype('Int64')

    if '연령' in df.columns:
        df['연령'] = (
            df['연령'].astype(str)
            .str.replace('대', '', regex=False)
            .pipe(pd.to_numeric, errors='coerce')
            .astype('Int64')
        )

    regions = [
        '서울', '부산', '대구', '인천', '광주', '대전', '울산', '세종',
        '경기', '강원', '충북', '충남', '전북', '전남', '경북', '경남', '제주'
    ]
    region_type = CategoricalDtype(categories=regions, ordered=True)
    for col in ['거주시도명', '직장시도명']:
        if col in df.columns:
            df[col] = (
                df[col].astype(str)
                .astype(region_type)
                .cat.codes.replace(-1, pd.NA)
            )
            
    count_cols = [
        '연회비발생카드수_B0M', '상품관련면제카드수_B0M',
        '임직원면제카드수_B0M', '우수회원면제카드수_B0M',
        '기타면제카드수_B0M'
    ]
    for col in count_cols:
        if col in df.columns:
            df[col] = (
                df[col].astype(str)
                .str.replace('개', '', regex=False)
                .pipe(pd.to_numeric, errors='coerce')
                .astype('Int64')
            )

    le_cols = ['_1순위신용체크구분', '_2순위신용체크구분', 'Life_Stage']
    for col in le_cols:
        if col in df.columns:
            df[col] = pd.factorize(df[col], sort=True)[0]

    return df

df1 = preprocess(pd.read_parquet('train/1.회원정보/201807_train_회원정보.parquet'))
df2 = preprocess(pd.read_parquet('train/1.회원정보/201808_train_회원정보.parquet'))
df3 = preprocess(pd.read_parquet('train/1.회원정보/201809_train_회원정보.parquet'))
df4 = preprocess(pd.read_parquet('train/1.회원정보/201810_train_회원정보.parquet'))
df5 = preprocess(pd.read_parquet('train/1.회원정보/201811_train_회원정보.parquet'))
df6 = preprocess(pd.read_parquet('train/1.회원정보/201812_train_회원정보.parquet'))

In [4]:
dfs = [df.drop(columns=['기준년월'], errors='ignore') for df in [df1, df2, df3, df4, df5, df6]]

def merge_two_avg(df_left, df_right):
    merged = pd.merge(df_left, df_right, on=['ID', 'Segment'], how='outer', suffixes=('_left', '_right'))
    result = merged[['ID', 'Segment']].copy()
    
    for col in set(df_left.columns).union(df_right.columns):
        if col in ['ID', 'Segment']:
            continue
        col_left = f"{col}_left" if f"{col}_left" in merged.columns else None
        col_right = f"{col}_right" if f"{col}_right" in merged.columns else None
        
        cols_to_avg = [c for c in [col_left, col_right] if c is not None]
        result[col] = merged[cols_to_avg].mean(axis=1, skipna=True)
    
    return result

from functools import reduce
merged_df = reduce(merge_two_avg, dfs)

# 5. 결과 확인
print(merged_df.shape)
print(merged_df.head())


(400000, 77)
             ID Segment  할인금액_기본연회비_B0M  이용가능카드수_체크  수신거부여부_DM  제휴연회비_B0M  \
0  TRAIN_000000       D             0.0         1.0        0.0        0.0   
1  TRAIN_000001       E             0.0         0.0        0.0        0.0   
2  TRAIN_000002       C             0.0         1.0        0.0        0.0   
3  TRAIN_000003       D             0.0         1.0        0.0        0.0   
4  TRAIN_000004       E             0.0         1.0        0.0        0.0   

   동의여부_한도증액안내  _2순위카드이용건수  이용금액_R3M_체크_가족  이용카드수_체크  ...  회원여부_이용가능_CA  \
0      1.00000         0.0             0.0       0.0  ...           1.0   
1      0.00000         0.0             0.0       0.0  ...           1.0   
2      0.96875         0.0             0.0       0.0  ...           1.0   
3      1.00000         0.0             0.0       0.0  ...           1.0   
4      0.00000         0.0             0.0       1.0  ...           1.0   

   할인금액_제휴연회비_B0M  이용금액_R3M_체크  유효카드수_신용체크    연령  이용카드수_체크_가족  수신거부여부_TM 

In [8]:
nan_columns = merged_df.columns[merged_df.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
['가입통신회사코드', '최종유효년월_신용_이용', '최종카드발급일자', '최종유효년월_신용_이용가능', '연령', '직장시도명']


In [10]:
ex1 = merged_df

In [11]:
cols_to_drop = ['가입통신회사코드', '최종유효년월_신용_이용', '최종카드발급일자', '최종유효년월_신용_이용가능', '연령', '직장시도명']
ex1.drop(columns=cols_to_drop, inplace=True)

In [12]:
missing_mask = ex1.isna() | (ex1 == -1)
missing_ratio = missing_mask.mean()

high_na = missing_ratio[missing_ratio > 0.2].index.tolist()

high_const_cols = []
threshold_const = 0.8

for col in ex1.columns:
    top_ratio = ex1[col].value_counts(normalize=True, dropna=False).values[0]
    if top_ratio > threshold_const:
        high_const_cols.append(col)

to_drop = list(set(high_na + high_const_cols))

if 'Segment' in to_drop:
    to_drop.remove('Segment')

print("삭제 대상 컬럼 (결측>20% 또는 동일값>80%):", to_drop)

ex1.drop(columns=to_drop, inplace=True)

삭제 대상 컬럼 (결측>20% 또는 동일값>80%): ['할인금액_기본연회비_B0M', '소지여부_신용', '회원여부_연체', '동의여부_한도증액안내', '제휴연회비_B0M', '탈회횟수_발급1년이내', '이용금액_R3M_체크_가족', '이용카드수_체크', '연회비발생카드수_B0M', '유효카드수_체크_가족', '청구금액_제휴연회비_B0M', '마케팅동의여부', '이용가능카드수_신용_가족', '_1순위신용체크구분', '유효카드수_신용_가족', '기타면제카드수_B0M', '기본연회비_B0M', '이용금액_R3M_신용_가족', '우수회원면제카드수_B0M', '임직원면제카드수_B0M', '탈회횟수_발급6개월이내', '카드신청건수', '청구금액_기본연회비_B0M', '이용카드수_신용_가족', '회원여부_이용가능_CA', '할인금액_제휴연회비_B0M', '이용금액_R3M_체크', '회원여부_이용가능', '상품관련면제카드수_B0M', '이용가능카드수_체크_가족', '이용카드수_체크_가족', '연회비할인카드수_B0M', '_2순위신용체크구분']


In [13]:
num_df = ex1.select_dtypes(include=[np.number]).dropna()

corr = num_df.corr().abs()

high_corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)
high_corr_pairs.columns = ['Feature_1', 'Feature_2', 'Correlation']
high_corr_pairs = high_corr_pairs[high_corr_pairs['Correlation'] > 0.8]

if not np.issubdtype(ex1['Segment'].dtype, np.number):
    segment_map = {label: idx for idx, label in enumerate(sorted(ex1['Segment'].unique()))}
    ex1['Segment_encoded'] = ex1['Segment'].map(segment_map)
else:
    ex1['Segment_encoded'] = ex1['Segment']

segment_corr = ex1[num_df.columns].corrwith(ex1['Segment_encoded']).abs()

high_corr_pairs['Corr_with_Segment_1'] = high_corr_pairs['Feature_1'].map(segment_corr)
high_corr_pairs['Corr_with_Segment_2'] = high_corr_pairs['Feature_2'].map(segment_corr)

high_corr_pairs = high_corr_pairs.sort_values(by='Correlation', ascending=False).reset_index(drop=True)

print(f"▶ 상관계수 0.7 초과 변수쌍 수: {len(high_corr_pairs)}")
display(high_corr_pairs)


▶ 상관계수 0.7 초과 변수쌍 수: 23


,Feature_1,Feature_2,Correlation,Corr_with_Segment_1,Corr_with_Segment_2
0,입회일자_신용,입회경과개월수_신용,0.999228,0.252289,0.252294
1,이용가능카드수_체크,유효카드수_체크,0.997035,0.147506,0.147918
2,수신거부여부_DM,수신거부여부_메일,0.978913,0.105715,0.107539
3,이용가능카드수_신용체크,유효카드수_신용체크,0.962551,0.358194,0.347391
4,_2순위카드이용건수,_2순위카드이용금액,0.954526,0.378077,0.401338
5,이용금액_R3M_신용,이용금액_R3M_신용체크,0.942598,0.598754,0.632131
6,이용가능여부_해외겸용_본인,보유여부_해외겸용_본인,0.934725,0.161883,0.147534
7,이용가능카드수_신용,유효카드수_신용,0.932848,0.359259,0.351201
8,이용금액_R3M_신용,_1순위카드이용금액,0.932375,0.598754,0.590759
9,이용카드수_신용체크,이용카드수_신용,0.930928,0.405206,0.383813


In [14]:
to_drop = []

for _, row in high_corr_pairs.iterrows():
    f1, f2 = row['Feature_1'], row['Feature_2']
    c1, c2 = row['Corr_with_Segment_1'], row['Corr_with_Segment_2']
    
    if pd.isna(c1) or pd.isna(c2):
        continue
    
    if c1 < c2:
        to_drop.append(f1)
    else:
        to_drop.append(f2)

to_drop = list(set(to_drop))

# 결과 출력
print(f"▶ 제거 대상 피처 수: {len(to_drop)}")
print("제거할 피처 목록:")
print(to_drop)

▶ 제거 대상 피처 수: 17
제거할 피처 목록:
['_2순위카드이용건수', '이용금액_R3M_신용', '이용가능카드수_체크', '보유여부_해외겸용_본인', '_1순위카드이용금액', '이용여부_3M_해외겸용_신용_본인', '이용가능카드수_신용', '소지카드수_유효_신용', '소지카드수_이용가능_신용', '이용가능여부_해외겸용_신용_본인', '수신거부여부_TM', '수신거부여부_DM', '입회일자_신용', '보유여부_해외겸용_신용_본인', '이용카드수_신용', '유효카드수_신용체크', '유효카드수_신용']


In [15]:
cols_to_drop = ['거주시도명', '보유여부_해외겸용_본인', '소지카드수_유효_신용', '이용가능여부_해외겸용_신용_본인', '이용금액_R3M_신용', '_2순위카드이용건수', '_1순위카드이용금액', '보유여부_해외겸용_신용_본인', '수신거부여부_DM', '이용여부_3M_해외겸용_신용_본인', '수신거부여부_메일', '유효카드수_신용체크', '이용가능카드수_신용', '유효카드수_신용', '소지카드수_이용가능_신용', '이용가능카드수_체크', '이용카드수_신용', '입회경과개월수_신용']
ex1.drop(columns=cols_to_drop, inplace=True)

In [18]:
cols_to_drop = ['Segment_encoded']
ex1.drop(columns=cols_to_drop, inplace=True)
cols = ex1.columns.tolist()
cols

['ID',
 'Segment',
 '이용거절여부_카드론',
 '탈회횟수_누적',
 '_1순위카드이용건수',
 '이용가능카드수_신용체크',
 '입회일자_신용',
 '이용가능여부_해외겸용_본인',
 '회원여부_이용가능_카드론',
 'Life_Stage',
 '이용금액_R3M_신용체크',
 '이용카드수_신용체크',
 '최종탈회후경과월',
 '수신거부여부_SMS',
 '이용여부_3M_해외겸용_본인',
 '최종카드발급경과월',
 '유효카드수_체크',
 '_2순위카드이용금액',
 '수신거부여부_TM',
 '남녀구분코드']

In [19]:
ex1.to_parquet('회원_전처리_Segment.parquet', index=False)

In [5]:
ddf1 = preprocess(pd.read_parquet('test/1.회원정보/201807_test_회원정보.parquet'))
ddf2 = preprocess(pd.read_parquet('test/1.회원정보/201808_test_회원정보.parquet'))
ddf3 = preprocess(pd.read_parquet('test/1.회원정보/201809_test_회원정보.parquet'))
ddf4 = preprocess(pd.read_parquet('test/1.회원정보/201810_test_회원정보.parquet'))
ddf5 = preprocess(pd.read_parquet('test/1.회원정보/201811_test_회원정보.parquet'))
ddf6 = preprocess(pd.read_parquet('test/1.회원정보/201812_test_회원정보.parquet'))

In [23]:

dfs = [ddf1, ddf2, ddf3, ddf4, ddf5, ddf6]
for i in range(len(dfs)):
    if '기준년월' in dfs[i].columns:
        dfs[i] = dfs[i].drop(columns=['기준년월'])

# ID 기준으로 병합 후 평균
from functools import reduce

merged_df = reduce(
    lambda left, right: pd.merge(left, right, on='ID', how='outer', suffixes=('', '_dup')),
    dfs
)

# 같은 이름의 열 평균 구하기
from collections import defaultdict
import pandas as pd

result = pd.DataFrame()
result['ID'] = merged_df['ID']

# 열 이름 모아 평균 구하기
col_dict = defaultdict(list)
for col in merged_df.columns:
    if col != 'ID':
        base_col = col.split('_dup')[0]
        col_dict[base_col].append(col)

for base_col, cols in col_dict.items():
    result[base_col] = merged_df[cols].mean(axis=1, skipna=True)

# 결과 확인
print(result.head())

           ID  남녀구분코드    연령  회원여부_이용가능  회원여부_이용가능_CA  회원여부_이용가능_카드론  소지여부_신용  \
0  TEST_00000     1.0  40.0        1.0           1.0            0.0      1.0   
1  TEST_00001     1.0  60.0        1.0           1.0            0.0      1.0   
2  TEST_00002     1.0  40.0        1.0           1.0            1.0      1.0   
3  TEST_00003     2.0  40.0        1.0           1.0            1.0      1.0   
4  TEST_00004     2.0  40.0        1.0           0.0            1.0      1.0   

   소지카드수_유효_신용  소지카드수_이용가능_신용     입회일자_신용  ...  할인금액_제휴연회비_B0M  \
0          2.0            2.0  20140501.0  ...             0.0   
1          1.0            1.0  20160201.0  ...             0.0   
2          2.0            2.0  20180301.0  ...             0.0   
3          1.0            1.0  20120701.0  ...             0.0   
4          1.0            1.0  20031201.0  ...             0.0   

   청구금액_기본연회비_B0M  청구금액_제휴연회비_B0M  상품관련면제카드수_B0M  임직원면제카드수_B0M  우수회원면제카드수_B0M  \
0             0.0             0.0        

In [26]:
cols = ['ID',
 '이용거절여부_카드론',
 '탈회횟수_누적',
 '_1순위카드이용건수',
 '이용가능카드수_신용체크',
 '입회일자_신용',
 '이용가능여부_해외겸용_본인',
 '회원여부_이용가능_카드론',
 'Life_Stage',
 '이용금액_R3M_신용체크',
 '이용카드수_신용체크',
 '최종탈회후경과월',
 '수신거부여부_SMS',
 '이용여부_3M_해외겸용_본인',
 '최종카드발급경과월',
 '유효카드수_체크',
 '_2순위카드이용금액',
 '수신거부여부_TM',
 '남녀구분코드']

result = result[cols]
result

,ID,이용거절여부_카드론,탈회횟수_누적,_1순위카드이용건수,이용가능카드수_신용체크,입회일자_신용,이용가능여부_해외겸용_본인,회원여부_이용가능_카드론,Life_Stage,이용금액_R3M_신용체크,이용카드수_신용체크,최종탈회후경과월,수신거부여부_SMS,이용여부_3M_해외겸용_본인,최종카드발급경과월,유효카드수_체크,_2순위카드이용금액,수신거부여부_TM,남녀구분코드
0,TEST_00000,1.0,1.000000,35.230769,2.0,20140501.0,1.0,0.0,4.000000,20527.230769,2.000000,102.884615,1.0,1.0,47.884615,0.0,4901.000000,1.0,1.0
1,TEST_00001,0.0,2.000000,32.692308,2.0,20160201.0,1.0,0.0,3.000000,18726.769231,2.000000,22.884615,0.0,1.0,21.884615,1.0,0.000000,0.0,1.0
2,TEST_00002,0.0,2.000000,128.615385,2.0,20180301.0,0.0,1.0,4.000000,38090.692308,2.000000,103.884615,0.0,0.0,7.884615,0.0,9661.884615,0.0,1.0
3,TEST_00003,0.0,0.000000,97.500000,1.0,20120701.0,1.0,1.0,4.000000,4609.423077,1.000000,0.000000,1.0,1.0,9.884615,0.0,0.000000,0.0,2.0
4,TEST_00004,0.0,0.000000,43.923077,3.0,20031201.0,1.0,1.0,4.961538,11493.000000,3.000000,0.000000,0.0,1.0,3.884615,1.0,6067.192308,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,TEST_99995,0.0,0.384615,-2.000000,0.0,20130301.0,0.0,0.0,1.000000,0.000000,0.000000,0.192308,0.0,0.0,5.038462,0.0,0.000000,0.0,2.0
99996,TEST_99996,0.0,0.000000,11.153846,1.0,20171001.0,0.0,1.0,6.000000,1152.538462,1.000000,0.000000,0.0,0.0,9.884615,0.0,0.000000,0.0,1.0
99997,TEST_99997,0.0,1.000000,2.846154,1.0,20180601.0,0.0,1.0,4.000000,0.000000,0.000000,20.884615,0.0,0.0,8.884615,0.0,0.000000,0.0,2.0
99998,TEST_99998,0.0,1.000000,188.538462,6.0,20120201.0,1.0,1.0,0.000000,154367.961538,5.153846,62.884615,1.0,1.0,1.961538,2.0,29768.346154,1.0,1.0


In [27]:
result.to_parquet('회원_전처리_test.parquet', index=False)

In [28]:
nan_columns = result.columns[result.isnull().any()].tolist()

print("NaN이 포함된 열 목록:")
print(nan_columns)

NaN이 포함된 열 목록:
[]
